# Jupyter Notebooks

**Part I · Visualization** — Tutorial 03

The `Visualizer` detects Jupyter/IPython automatically and adapts: `show()`
renders inline instead of opening a browser tab, `run()` is unavailable (it
would block the kernel), and `display_snapshot()` produces serverless embeds.
This chapter walks through the notebook workflows for developing an idea step by
step.

> **Prerequisites:** [Tutorial 02](../02_use_cases/) (the methods for adding and
> updating objects) and the `Visualizer` basics in
> [Tutorial 04](../04_getting_started/).


## Setup


In [ ]:
from pytanga.geometry import Direction, Line, Point, Sphere
from pytanga.viz import Visualizer


## 1. Auto-detection

In a notebook, the visualizer:

- sets `open_browser=False` (no popup),
- makes `run()` unavailable (it would block the kernel indefinitely),
- uses `start_server()` / `flush()` / `stop_server()` (or `show()`) for the
  non-blocking workflow, and
- renders an inline `<iframe>` via `_repr_html_()` when the `Visualizer` object
  is the last expression in a cell.


## 2. Live inline workflow

Live display embeds the **running** server in an inline iframe, so you can
rotate, pan, zoom, and animate. The pattern is: start the server, add and flush,
then put the visualizer as the last expression of a cell to render the iframe.


In [ ]:
# Cell 1 — start the server
viz = Visualizer()
viz.start_server()
print("viewer available at", viz.url)


In [ ]:
# Cell 2 — add entities and flush
viz.add(Point(2, 0, 0), color="#ff4444", label="P1")
viz.add(Sphere(Point(0, 0, 0), radius=2.5), wireframe=True, opacity=0.3)
viz.flush()
print("entities added and flushed")


In [ ]:
# Cell 3 — render inline (last expression)
viz


In [ ]:
# Cell 4 — add more later
viz.add(Line(origin=Point(0, 0, 0), direction=Direction(1, 0, 0)), color="#44ff44")
viz.flush()
viz  # re-render the iframe


In [ ]:
# Cell 5 — cleanup
viz.stop_server()
print("server stopped")


## 3. Idempotent `show()` / `display()` and context managers

Re-running a cell that calls `show()` or `display()` does **not** open a second
viewer — it flushes the latest state into the already-open one. The viewer is
identified by an optional `viewer_name`; otherwise the current notebook cell id
is used (falling back to the scene name).


In [ ]:
viz = Visualizer()
viz.add(Point(1, 2, 3))
viz.show()          # opens the inline viewer (starts the server)
viz.add(Point(4, 5, 6))
viz.show()          # no new viewer — just flushes the update


In [ ]:
# Context managers: clear + show on entry, flush on exit
with viz:
    viz.add(Point(1, 2, 3))

with viz.scene("detail"):
    viz.scene("detail").add(Point(4, 5, 6))


In [ ]:
# Keep two cells pointed at the same scene with an explicit viewer_name
viz.scene("detail").display(viewer_name="cell-a")
viz.scene("detail").display(viewer_name="cell-b")


## 4. Side-by-side scenes — `display_row()`

`display_row()` shows several scenes side by side in one cell, in **live** mode
(each is a running iframe) or **static** mode (serverless snapshots).


In [ ]:
overview = viz.scene("overview")
detail = viz.scene("detail")

overview.add(Sphere(Point(0, 0, 0), radius=3), opacity=0.2)
detail.add(Sphere(Point(2, 1, 0), radius=1), opacity=0.8)

viz.flush()
viz.display_row((overview, "left"), (detail, "right"), width="100%", height=500, gap=12)


## 5. Static, serverless snapshots

`display_snapshot()` renders the current scene as a self-contained, serverless
Three.js document inline — no WebSocket server, no daemon threads. Call it after
each `add()` to build a figure progressively.


In [ ]:
viz = Visualizer()
viz.add(Point(1, 2, 3), color="#ff4444")
viz.display_snapshot()

viz.add(Sphere(Point(0, 0, 0), radius=2), opacity=0.3)
viz.display_snapshot()


In [ ]:
# Static side-by-side snapshots
viz.display_row((overview, None), (detail, None), mode="static")


## 6. Typical notebook use cases

| Situation | Recommended |
|---|---|
| Rotate / zoom / animate live | `start_server()` + `flush()` + cell `_repr_html_` |
| Quick static snapshot | `display_snapshot()` |
| Progressive figure building | call `display_snapshot()` after each `add()` |
| Export to HTML/PDF (nbconvert) | `display_snapshot()` |
| One-off demo, no animation | context manager |
| One-off demo, animation | `animate(auto_clear=True)` |
| Long-running animation | pre-create with `viz(...)` and update `.entity` in place |


## 7. Limitations

- **Remote Jupyter** (Colab, Binder, remote kernels): the iframe points to
  `localhost`, which is the server machine, not your browser — open the printed
  URL in a tab on the machine running the kernel.
- **Port conflicts:** `start_server()` defaults to port 8765; pass `port=...` or
  `port=0` to auto-pick a free port.
- **Multiple scenes:** create named scenes with `viz.scene("name")` instead of
  multiple `Visualizer` instances — all scenes share one server on one port.


## Visual Examples

A standalone HTML figure exported via `export_snapshot()`.


In [ ]:
viz = Visualizer(title="Tanga — Notebook scene")
viz.add(Point(2, 1, 0), color="#ff4444", label="$P$")
viz.add(Sphere(Point(0, 0, 0), 2.0), wireframe=True, opacity=0.3)
viz.add(Line(origin=Point(0, 0, 0), direction=Direction(1, 0, 0)), color="#44ff44")
viz.export_snapshot("_output/03_jupyter_figure.html", overwrite=True)
print("figure written")


## Summary

| Task | API |
|---|---|
| Auto-detect notebook | automatic (`show()` renders inline) |
| Live inline viewer | `start_server()` / `flush()` / last-expression `viz` |
| Idempotent re-render | repeated `show()` / `display()` |
| Clear + show, flush on exit | `with viz:` / `with viz.scene("name"):` |
| Side-by-side scenes | `viz.display_row(...)` (live or static) |
| Static snapshot | `viz.display_snapshot()` |
| Cleanup | `viz.stop_server()` |

**Next:** [04 — Getting Started](../04_getting_started/).
